# Bank Customer Churn Prediction
A Business Intelligence pipeline to predict customer attrition and identify key drivers of churn.


In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib
!curl -L -o bank-customer-churn.zip https://www.kaggle.com/api/v1/datasets/download/radheshyamkollipara/bank-customer-churn
!unzip -q -o bank-customer-churn.zip -d data/


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score


### 1. Data Cleaning & Feature Engineering
Removes irrelevant identifiers and encodes categorical business features.


In [ ]:
import glob
file_path = glob.glob('data/*.csv')[0]
df = pd.read_csv(file_path)

# Drop unhelpful identifiers
df = df.drop(columns=['RowNumber', 'CustomerId', 'Surname'], errors='ignore')

# Encode categoricals (Geography, Gender)
for col in df.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

X = df.drop(columns=['Exited'], errors='ignore')
y = df['Exited'] if 'Exited' in df.columns else df.iloc[:, -1]

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)


### 2. Model Training


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
model.fit(X_train, y_train)


### 3. Business Intelligence: Feature Importance
Identifying the primary reasons customers leave.


In [ ]:
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Plot Feature Importance
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
plt.title("Drivers of Customer Churn")
plt.bar(range(X.shape[1]), importances[indices], align="center")
plt.xticks(range(X.shape[1]), X.columns[indices], rotation=45, ha='right')
plt.tight_layout()
plt.show()
